In [2]:
# Import section
from backtesting import Backtest
import pandas as pd
from backtesting import Strategy
from backtesting.lib import crossover
import numpy as np
import math

# META
MARGIN = 0.01
CASH = 10_000
COMMISSION = 0.00002
table_name = 'f0_trailing_sl'

# DEFINE
trade_histories = pd.DataFrame({
  "datetime": [],
  "result": []
})

# add a row
# df.loc[len(df)] = ['Bob', 30]

/home/chris/backtester/dema_strat/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

# Strategy

In [3]:
# Define the EMA Strategy
class STRAT(Strategy):
    n1 = 9  # Fast EMA period
    n2 = 21  # Slow EMA period

    risk_pct = 0.01      # Risk 1% of account per trade
    trailing_sl_step = 1 # trailing sl step

    def init(self):
        super().init()

        # Calculate EMAs using Pandas directly
        close = pd.Series(self.data.Close)

        self.ema1 = self.I(lambda: close.ewm(span=self.n1, adjust=False).mean(), name="ema fast")
        self.ema2 = self.I(lambda: close.ewm(span=self.n2, adjust=False).mean(), name="ema slow")

    def next(self):
        price = self.data.Close[-1]

        prev_low  = self.data.Low[-2]
        prev_high = self.data.High[-2]

        long_signal  = crossover(self.ema1, self.ema2) 
        short_signal = crossover(self.ema2, self.ema1)

        risk_amt = max(self.equity, CASH) * self.risk_pct

        def get_size_by_leveraged(sl_dist):
            ideal_size = risk_amt / sl_dist
            # We subtract a buffer 85%
            max_leverage_size = (self.equity * 0.85) / (price * MARGIN)
            size = int(min(ideal_size, max_leverage_size))
            return size

        # trailing sl handle
        for trade in self.trades:
            sl_dist = abs(trade.sl - trade.entry_price)
            dist_to_sl = abs(price - trade.sl)

            sl_slice = sl_dist * self.trailing_sl_step

            if trade.is_long and dist_to_sl >= 2 * sl_slice and price >= trade.entry_price + sl_slice:
                trade.sl += sl_slice

            if trade.is_short and dist_to_sl >= 2 * sl_slice and price >= trade.entry_price + sl_slice:
                trade.sl -= sl_slice

        if long_signal:

            sl_dist = abs(price - prev_low)

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return

            self.position.close()
            self.buy(size=size, sl=price - sl_dist)

        elif short_signal:

            sl_dist = abs(price - prev_high)

            if sl_dist == 0:
                return

            size = get_size_by_leveraged(sl_dist)

            if size <= 0:
                return


            self.position.close()
            self.sell(size=size, sl=price + sl_dist)


# Basic backtest

In [ ]:
# Load sample data and run a basic backtest
file_path = "~/market_data/XAUUSD/XAUUSD_M5.csv"

try:
  # Load data
  data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
  
  # Run backtest
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
  stats = bt.run()
  
  print(stats)
  
except Exception as e:
  print(f"Error: {str(e)}")

Start                     2020-01-02 01:00:00
End                       2025-12-31 23:55:00
Duration                   2190 days 22:55:00
Exposure Time [%]                     3.53781
Equity Final [$]                     16.01171
Equity Peak [$]                    22564.4508
Commissions [$]                    7945.90429
Return [%]                          -99.83988
Buy & Hold Return [%]               184.03997
Return (Ann.) [%]                   -65.36144
Volatility (Ann.) [%]                30.45747
CAGR [%]                            -65.80544
Sharpe Ratio                         -2.14599
Sortino Ratio                        -1.01644
Calmar Ratio                         -0.65408
Alpha [%]                           -88.38065
Beta                                 -0.06226
Max. Drawdown [%]                   -99.92904
Avg. Drawdown [%]                    -9.12628
Max. Drawdown Duration     2178 days 20:20:00
Avg. Drawdown Duration      109 days 13:04:00
# Trades                          

# SQLite connection

In [5]:
import sqlite3

# Setup SQLite database
db_path = "../backtest_results.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database setup complete at {db_path}")

conn.commit()

Database setup complete at ../backtest_results.db


# Combination test

In [11]:
from itertools import product
from collections import namedtuple


cursor.execute(f'''
CREATE TABLE IF NOT EXISTS {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    risk_pct REAL NOT NULL,
    trailing_sl_step REAL NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# Test all asset/timeframe combinations
assets = ['XAUUSD', 'EURUSD', 'GBPUSD', 'USDCAD', 'AUDUSD', 'USDCHF', 'US100', 'US500']
timeframes = ['M1', 'M5', 'M15', 'H1', 'H4', 'D1']

# Strategy parameters (from current STRAT class)
init_params = {
    'n1': 9,
    'n2': 21,
    'risk_pct': 0.01,
    'trailing_sl_step': 1
}

parameters = [
    timeframes,
    assets,
    [0.5, 1], # trailing sl step
    [0.005, 0.01] # risk percent
]

total_combinations = math.prod(len(p) for p in parameters)
print(f"Testing {total_combinations} combinations...")

for timeframe, asset, tsl_step, risk_pct in product(*parameters):
    file_path = f"~/market_data/{asset}/{asset}_{timeframe}.csv"
    
    try:
        # Load data
        data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
        
        # Run backtest
        bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
        stats = bt.run(trailing_sl_step=tsl_step, risk_pct=risk_pct)
        
        # Extract metrics
        return_pct = stats['Return [%]']
        max_drawdown_pct = stats['Max. Drawdown [%]']
        cagr_pct = stats['CAGR [%]']
        sharpe_ratio = stats['Sharpe Ratio']
        profit_factor = stats['Profit Factor']
        win_rate_pct = stats['Win Rate [%]']
        num_trades = stats['# Trades']
        equity_final = stats['Equity Final [$]']
        
        # Store in database
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, trailing_sl_step, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, tsl_step, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final))
        
        conn.commit()
        
        print(f"✓ {asset}_{timeframe}: Return={return_pct:.2f}%, DD={max_drawdown_pct:.2f}%, Sharpe={sharpe_ratio:.2f}, WinRate={win_rate_pct:.2f}%")
        
    except Exception as e:
        print(f"✗ {asset}_{timeframe}: Error - {str(e)}")
        # Store error record
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, trailing_sl_step, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, tsl_step))
        conn.commit()

print("\nBacktesting complete!")
print(f"Results stored in {db_path}")

Testing 192 combinations...
✓ XAUUSD_M1: Return=-99.86%, DD=-99.88%, Sharpe=-0.08, WinRate=18.07%
✓ XAUUSD_M1: Return=-99.83%, DD=-99.89%, Sharpe=-2.64, WinRate=18.64%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.91%, Sharpe=-0.15, WinRate=19.21%
✓ XAUUSD_M1: Return=-99.88%, DD=-99.94%, Sharpe=-2.50, WinRate=19.44%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.07, WinRate=17.24%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.03, WinRate=17.23%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.07, WinRate=17.70%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-9.23, WinRate=17.69%
✓ GBPUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.34, WinRate=16.80%
✓ GBPUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.50, WinRate=16.64%
✓ GBPUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-1.94, WinRate=18.02%
✓ GBPUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.34, WinRate=17.40%
✓ USDCAD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.42, WinRate=14.05%
✓ USDCAD_M1: Return=-100.00%, DD=-100.00%, S

/tmp/ipykernel_16198/2246970809.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-99.10%, DD=-99.38%, Sharpe=-1.10, WinRate=18.40%


/tmp/ipykernel_16198/2246970809.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-98.76%, DD=-99.53%, Sharpe=-0.63, WinRate=18.64%


...

✓ AUDUSD_D1: Return=-5.38%, DD=-13.72%, Sharpe=-0.27, WinRate=14.71%
✓ AUDUSD_D1: Return=-10.94%, DD=-26.05%, Sharpe=-0.28, WinRate=14.71%
✓ USDCHF_D1: Return=-4.85%, DD=-13.75%, Sharpe=-0.17, WinRate=13.64%
✓ USDCHF_D1: Return=-9.70%, DD=-27.51%, Sharpe=-0.16, WinRate=13.64%
✓ USDCHF_D1: Return=-6.43%, DD=-14.25%, Sharpe=-0.23, WinRate=12.50%
✓ USDCHF_D1: Return=-12.86%, DD=-28.50%, Sharpe=-0.21, WinRate=12.50%


/tmp/ipykernel_16198/2246970809.py:57: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(trailing_sl_step=tsl_step, risk_pct=risk_pct)
/tmp/ipykernel_16198/2246970809.py:57: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(trailing_sl_step=tsl_step, risk_pct=risk_pct)
/tmp/ipykernel_16198/2246970809.py:57: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(trailing_sl_step=tsl_step, risk_pct=risk_pct)
/tmp/ipykernel_16198/2246970809.py:57: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run(trailing_sl_step=tsl_step, risk_pct=risk_pct)
/tmp/ipykernel_1

✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US500_D1: Return=-1.16%, DD=-6.93%, Sharpe=-0.08, WinRate=0.00%
✓ US500_D1: Return=14.10%, DD=-9.36%, Sharpe=0.32, WinRate=21.43%
✓ US500_D1: Return=-0.90%, DD=-6.69%, Sharpe=-0.06, WinRate=0.00%


/tmp/ipykernel_16198/2246970809.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_16198/2246970809.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_16198/2246970809.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis 

✓ US500_D1: Return=13.71%, DD=-10.13%, Sharpe=0.31, WinRate=21.43%

Backtesting complete!
Results stored in ../backtest_results.db


### Summary

In [12]:
query_pf = f"""
SELECT asset, timeframe, trailing_sl_step, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE profit_factor IS NOT NULL and num_trades >= 1000
ORDER BY profit_factor DESC
LIMIT 20
"""
top_pf = pd.read_sql_query(query_pf, conn)

query_wr = f"""
SELECT asset, timeframe, trailing_sl_step, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE win_rate_pct IS NOT NULL and num_trades >= 1000
ORDER BY win_rate_pct DESC
LIMIT 20
"""
top_wr = pd.read_sql_query(query_wr, conn)

print("Top 20 by Profit Factor")
print(top_pf.to_string(index=False))
print(f"Average profit factor (top 20): {top_pf['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20 by profit factor): {top_pf['win_rate_pct'].mean():.2f}%")
print("\nTop 20 by Win Rate")
print(top_wr.to_string(index=False))
print(f"Average profit factor (top 20 by win rate): {top_wr['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20): {top_wr['win_rate_pct'].mean():.2f}%")


Top 20 by Profit Factor
 asset timeframe  trailing_sl_step  return_pct  profit_factor  win_rate_pct  num_trades  equity_final
 US100       M15               0.5  -97.955535       1.100432     12.620773        3312    204.446527
 US100       M15               1.0  -97.171535       1.075979     13.014957        3811    282.846486
XAUUSD        H1               0.5   26.554442       1.025714     17.450980        1530  12655.444179
XAUUSD        H1               1.0   22.126738       1.022792     18.300654        1530  12212.673781
 US100        M1               1.0  -98.891235       1.019775     19.585879        2946    110.876548
XAUUSD        H1               1.0   13.805738       1.014762     18.229508        1525  11380.573755
XAUUSD        H1               0.5   15.731152       1.011824     17.377049        1525  11573.115216
 US100        M1               0.5  -98.763965       1.003512     18.644597        3202    123.603512
 US500        H1               0.5   -8.104376       1.002

### Post-process

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

query = f"""
SELECT max_drawdown_pct, win_rate_pct
FROM {table_name}
WHERE max_drawdown_pct IS NOT NULL AND win_rate_pct IS NOT NULL
"""
df_stats = pd.read_sql_query(query, conn)

mean_dd = df_stats['max_drawdown_pct'].mean()
mean_wr = df_stats['win_rate_pct'].mean()

print(f"Mean max drawdown (%): {mean_dd:.4f}")
print(f"Mean win rate (%): {mean_wr:.4f}")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Distribution of Max Drawdown (%)", "Distribution of Win Rate (%)"]
)

fig.add_trace(go.Histogram(x=df_stats["max_drawdown_pct"], name="Max Drawdown", nbinsx=40), row=1, col=1)
fig.add_trace(go.Histogram(x=df_stats["win_rate_pct"], name="Win Rate", nbinsx=40), row=1, col=2)

fig.update_layout(
    title="Interactive Backtest Distributions",
    bargap=0.1,
    template="plotly_white"
)
fig.show()

Mean max drawdown (%): -72.0365
Mean win rate (%): 16.9510


In [ ]:
bt.plot(open_browser=False, filename=f"{table_name}.html", resample=False, plot_pl=False, plot_volume=False)